In [ ]:
import sys, os, tempfile, warnings
from pathlib import Path
import torch

NOTEBOOK_DIR = Path().resolve()                    # tests/
SRC_DIR      = NOTEBOOK_DIR.parent / "src"         # src/

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

# ── Imports ───────────────────────────────────────────────────────────────────
from dataloaders._ts_dataloader import DataLoaderFactory
from dataloaders.ts_sharding import (
    write_sharded_dataset,
)
from common._base_model import BaseModel
from common.train import train, train_distributed, eval_test
from models.linear import TinyLinearModel
from _test_utils import make_df, make_mcfg, make_entry, make_dcfg, setup_test_data, remove_test_data

TEST_DATA_DIR = os.path.join(os.getcwd(), "test_data")

warnings.filterwarnings("ignore")
print("imports OK")

## Test — `train()`, resume from checkpoint

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    csv_path = f"{TEST_DATA_DIR}/data.csv"
    make_df(n_series=3, n_steps=300).to_csv(csv_path, index=False)

    mcfg = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=10, checkpoint_dir=TEST_DATA_DIR,
    )
    entry = make_entry(csv_path, name="resume_ds", horizon=6, val_size=50, test_size=50)
    dcfg  = make_dcfg([entry])

    # First run — saves final.pt
    factory      = DataLoaderFactory(mcfg, dcfg)
    train_loader = factory.train_dataloader()
    val_loaders  = factory.val_dataloaders()
    model = TinyLinearModel(mcfg)
    train(model, mcfg, train_loader, val_loaders, device=torch.device("cpu"))
    step_after_first = model.global_step
    print(f"after first run: global_step={step_after_first}")

    # Second run — resume from final.pt, train 10 more steps
    mcfg2 = make_mcfg(
        context_length=32, fcd_samples=4, batch_size=2,
        max_steps=step_after_first + 10, checkpoint_dir=TEST_DATA_DIR,
    )
    factory2     = DataLoaderFactory(mcfg2, dcfg)
    model2       = TinyLinearModel(mcfg2)
    train(
        model2, mcfg2,
        factory2.train_dataloader(), factory2.val_dataloaders(),
        device  = torch.device("cpu"),
        resume  = f"{TEST_DATA_DIR}/final.pt",
    )
    print(f"after resume:    global_step={model2.global_step}")
    assert model2.global_step == step_after_first + 10, \
        f"expected {step_after_first + 10}, got {model2.global_step}"
    print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)

## Test — `train_distributed()`, sharded dataset, mp.spawn

In [ ]:
try:
    setup_test_data(TEST_DATA_DIR)
    # Use gloo backend so this works on CPU / machines without NCCL
    BACKEND    = "nccl" if torch.cuda.is_available() else "gloo"
    WORLD_SIZE = min(2, torch.cuda.device_count() or 2)
    print(f"backend={BACKEND}  world_size={WORLD_SIZE}")

    shard_dir = f"{TEST_DATA_DIR}/sharded"
    csv_path  = f"{TEST_DATA_DIR}/data.csv"
    CTX     = 32
    HORIZON = 6

    df = make_df(n_series=3, n_steps=600, seed=99)
    df.to_csv(csv_path, index=False)

    write_sharded_dataset(
        df             = df,
        out_dir        = shard_dir,
        val_size       = 80,
        test_size      = 80,
        context_length = CTX,
        shard_size     = 100,   # small shards — ensures each rank gets ≥1 shard
    )

    shard_files = sorted(Path(shard_dir).glob("shard_*.parquet"))
    print(f"{len(shard_files)} train shards written")
    assert len(shard_files) >= WORLD_SIZE, \
        f"need ≥{WORLD_SIZE} shards for {WORLD_SIZE} ranks, got {len(shard_files)}"

    mcfg = make_mcfg(
        context_length  = CTX,
        fcd_samples     = 4,
        batch_size      = 2,
        max_steps       = 20,
        checkpoint_dir  = TEST_DATA_DIR,
    )
    entry = make_entry(
        path        = csv_path,
        name        = "dist_ds",
        horizon     = HORIZON,
        val_size    = 80,
        test_size   = 80,
        sharded_dir = shard_dir,
    )
    dcfg = make_dcfg([entry])

    # Factory is created BEFORE spawn — shared config, datasets rebuilt per rank
    factory = DataLoaderFactory(mcfg, dcfg)
    model   = TinyLinearModel(mcfg)

    train_distributed(
        model      = model,
        mcfg       = mcfg,
        factory    = factory,
        backend    = BACKEND,
        use_spawn  = True,
        world_size = WORLD_SIZE,
        seed       = 42,
    )

    assert Path(TEST_DATA_DIR, "final.pt").exists(), "final.pt not saved by rank 0"
    print("final.pt saved ✓")

    # Load weights and run test on single GPU
    test_model = TinyLinearModel(mcfg)
    factory_test = DataLoaderFactory(mcfg, dcfg)
    BaseModel.load_weights(f"{TEST_DATA_DIR}/final.pt", test_model)
    test_model.setup_inference(mcfg, device=torch.device("cpu"))
    preds = eval_test(test_model, factory_test, device=torch.device("cpu"))

    assert "dist_ds" in preds, f"missing keys in preds: {preds.keys()}"
    print("\n✓ TEST PASSED")
finally:
    remove_test_data(TEST_DATA_DIR)